# DO NOT USE - superseded by `panvram_loader_A100_v4.ipynb`

Its measurement cell swept the window sizes in a fixed order, 8 KiB first, right after the cohort was pushed to
the card. The 8 KiB batch then came out 3.0x more expensive per decoded block than the 32 KiB batch measured
second, which is an artefact of the order, not of the window size. v4 ramps the card first, shuffles the order,
repeats the sweep three times and prints microseconds per decoded block.


# panvram as a window loader for DNA model training: GPU path (v3)

Same protocol as the CPU rows of [`logs/loader-2026-10-22/`](https://github.com/yasha1971-coder/panvram/tree/main/logs/loader-2026-10-22):
batches of 1 024 windows, W = 8 192 and 32 768, 3 warm-up batches then 9 timed batches, median. The result is a
`uint8 [1024, W]` tensor already on the card, so no host-to-device copy is inside the timing.

What this notebook does **not** do: it does not compare with GenVarLoader or pyfaidx (those are the CPU rows), and it
does not claim its contents match a variant-based loader - panvram returns the assembly's own bases.

Runtime: **A100, High-RAM**. `N_ASM = 558` needs about 14 GB of VRAM and an 8.1 GB download; `N_ASM = 5` needs about
3.2 GB of VRAM and a 100 MB download.

**The five-assembly mode is not the five assemblies of the CPU rows.** The record ships the archives only inside a
tar, ordered by name, so the first five complete archives are release-2 haplotypes (`r2_HG00097_hap1`, ...), while the
CPU rows used five year-1 haplotypes. Cohort size is what this switch controls; the names differ.

Data: Zenodo record 10.5281/zenodo.23232317; reference T2T-CHM13v2.0 per DATASET.md. HPRC assemblies come from
BioProject PRJNA730823; their use here implies no endorsement by the HPRC.

In [ ]:
N_ASM = 558          # 558 or 5
BATCH = 1024
WS = (8192, 32768)
RUNS = 9
SEED = 20261003

print("\n" + "=" * 52)
print(f"====   N_ASM = {N_ASM}   ".ljust(52, "="))
print("=" * 52 + "\n")
if N_ASM not in (5, 558):
    raise SystemExit("N_ASM must be 5 or 558")
print(f"download {'about 8.1 GB (16 parts)' if N_ASM == 558 else 'about 100 MB (one range request)'}; "
      f"VRAM {'about 14 GB' if N_ASM == 558 else 'about 3.2 GB'}")

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!free -g | head -2
!nproc

In [ ]:
# panvram: the published repository, CUDA extension built against Colab's torch
!apt-get -qq install -y libzstd-dev > /dev/null
!git clone -q https://github.com/yasha1971-coder/panvram.git /content/panvram || true
%cd /content/panvram
!git log --oneline -1
!pip install -q -e . 2>&1 | tail -2
import panvram, torch
print('panvram', getattr(panvram, '__version__', 'ok'), 'with_cuda', panvram.with_cuda)
print('torch', torch.__version__, torch.cuda.get_device_name(0))
if not panvram.with_cuda:
    raise SystemExit('the CUDA extension did not build - check nvcc and that torch is a CUDA build')

In [ ]:
# Cohort fetch and verification. Identical to logs/loader-2026-10-22/scripts/colab_prepare.py,
# which is what was tested locally before this notebook was published.
# SPDX-License-Identifier: MIT
"""Fetch and verify a panvram q16k cohort from the Zenodo record, for the window-loader measurement.

The record (10.5281/zenodo.23232317) ships the 558 archives as one tar cut into 500 MiB parts; the individual
`*.q16k.rr3` files are NOT addressable on their own (they return 404). So:

  n_asm == 558: every part, checked against PARTS.sha256, joined, the whole tar checked against its .sha256,
                extracted, every archive checked against SHA256SUMS -> 558 archives.
  n_asm == 5:   the first 100 MB of part00 by HTTP range, the tar headers walked by hand, the archives that are
                complete inside that prefix extracted, the first five kept and checked against SHA256SUMS.q16k.

Either way: an archive of 0 bytes, a missing archive or a wrong SHA-256 raises RuntimeError at once.

Note for the five-assembly mode: the tar is ordered `./`, `./SHA256SUMS`, then the archives by name, so the five are
the first five of the release-2 assemblies (r2_HG00097_hap1, ...), NOT the five year-1 haplotypes of the CPU rows of
logs/loader-2026-10-22. The cohort size is what this mode controls; the names differ, and the report must say so.
"""
import hashlib
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

REC = "https://zenodo.org/records/23232317/files"
PREFIX_BYTES = 100 * 1024 * 1024
BLOCK = 512


def _md5(path, buf=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(buf)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _sha256(path, buf=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(buf)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _fetch(url, dest, byte_range=None, tries=5):
    dest = Path(dest)
    for attempt in range(1, tries + 1):
        cmd = ["curl", "-fsS", "--retry", "3", "--retry-delay", "5", "-o", str(dest)]
        if byte_range:
            cmd += ["-r", byte_range]
        r = subprocess.run(cmd + [url], capture_output=True, text=True)
        if r.returncode == 0 and dest.exists() and dest.stat().st_size > 0:
            return dest
        if attempt == tries:
            raise RuntimeError(f"download failed after {tries} attempts: {url}\n"
                               f"curl said: {r.stderr.strip() or r.returncode}; "
                               f"size on disk: {dest.stat().st_size if dest.exists() else 'no file'}")
    raise AssertionError


def _read_sums(path):
    sums = {}
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        digest, name = line.split(None, 1)
        sums[os.path.basename(name.strip())] = digest
    return sums


def _verify(dirpath, sums, expect_n):
    dirpath = Path(dirpath)
    got = sorted(p for p in dirpath.glob("*.q16k.rr3"))
    if len(got) != expect_n:
        raise RuntimeError(f"expected {expect_n} archives in {dirpath}, found {len(got)}")
    for p in got:
        if p.stat().st_size == 0:
            raise RuntimeError(f"{p.name} is 0 bytes - the download did not deliver the file")
        want = sums.get(p.name)
        if want is None:
            raise RuntimeError(f"{p.name} is not listed in the checksum file")
        have = _sha256(p)
        if have != want:
            raise RuntimeError(f"{p.name}: SHA-256 {have} does not match the record's {want}")
    print(f"VERIFIED\t{len(got)} archives, every SHA-256 equals the record's, no empty file")
    for p in got[:3]:
        print(f"  {p.name}\t{p.stat().st_size} B\tsha256 {_sha256(p)}\tmd5 {_md5(p)}")
    if len(got) > 3:
        print(f"  ... {len(got) - 3} more")
    return got


def _walk_tar_prefix(buf_path, out_dir, want):
    """Extract the archives wholly contained in a truncated tar, in tar order, up to `want` of them."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    taken, members = [], []
    with open(buf_path, "rb") as f:
        size = Path(buf_path).stat().st_size
        off = 0
        while off + BLOCK <= size and len(taken) < want:
            f.seek(off)
            head = f.read(BLOCK)
            if head[:1] == b"\0":
                break
            name = head[0:100].rstrip(b"\0").decode("utf-8", "replace")
            raw = head[124:136].rstrip(b"\0 ").decode() or "0"
            msize = int(raw, 8)
            data_off = off + BLOCK
            padded = (msize + BLOCK - 1) // BLOCK * BLOCK
            members.append((name, msize))
            base = os.path.basename(name)
            if base.endswith(".q16k.rr3"):
                if data_off + msize > size:
                    break                                    # truncated member: stop, do not write a short file
                f.seek(data_off)
                (out_dir / base).write_bytes(f.read(msize))
                taken.append(base)
            elif base == "SHA256SUMS" and data_off + msize <= size:
                f.seek(data_off)
                (out_dir / "SHA256SUMS.in_tar").write_bytes(f.read(msize))
            off = data_off + padded
    print(f"PREFIX\t{len(members)} tar members seen, {len(taken)} archives complete: {', '.join(taken)}")
    if len(taken) < want:
        raise RuntimeError(f"only {len(taken)} complete archives in the first {PREFIX_BYTES} bytes, need {want}; "
                           f"raise PREFIX_BYTES")
    return taken


def prepare_cohort(n_asm, dest="cohort", work="dl"):
    """Put a verified q16k cohort of n_asm assemblies in `dest`. Returns the list of archive paths."""
    if n_asm not in (5, 558):
        raise ValueError("n_asm must be 5 or 558")
    dest, work = Path(dest), Path(work)
    dest.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)

    if n_asm == 558:
        parts = [f"cohort558_q16k.tar.part{i:02d}" for i in range(16)]
        _fetch(f"{REC}/PARTS.sha256", work / "PARTS.sha256")
        psums = _read_sums(work / "PARTS.sha256")
        for p in parts:
            q = work / p
            if not (q.exists() and q.stat().st_size and _sha256(q) == psums.get(p)):
                _fetch(f"{REC}/{p}", q)
                have = _sha256(q)
                if have != psums.get(p):
                    raise RuntimeError(f"{p}: SHA-256 {have} does not match PARTS.sha256 {psums.get(p)}")
            print(f"  part ok {p} {q.stat().st_size} B", flush=True)
        tar = work / "cohort558_q16k.tar"
        with open(tar, "wb") as out:
            for p in parts:
                with open(work / p, "rb") as f:
                    shutil.copyfileobj(f, out, 8 << 20)
        _fetch(f"{REC}/cohort558_q16k.tar.sha256", work / "tar.sha256")
        want_tar = open(work / "tar.sha256").read().split()[0]
        have_tar = _sha256(tar)
        if have_tar != want_tar:
            raise RuntimeError(f"joined tar SHA-256 {have_tar} does not match the record's {want_tar}")
        print(f"TAR\t{tar.stat().st_size} B, SHA-256 matches the record", flush=True)
        with tarfile.open(tar) as t:
            t.extractall(dest, filter="data")
        for p in parts:
            (work / p).unlink(missing_ok=True)
        tar.unlink(missing_ok=True)
        inner = dest / "SHA256SUMS"
        if not inner.exists():
            raise RuntimeError("the tar did not contain SHA256SUMS")
        return _verify(dest, _read_sums(inner), 558)

    prefix = work / "part00.prefix"
    _fetch(f"{REC}/cohort558_q16k.tar.part00", prefix, byte_range=f"0-{PREFIX_BYTES - 1}")
    print(f"RANGE\t{prefix.stat().st_size} B of part00", flush=True)
    _walk_tar_prefix(prefix, dest, want=5)
    _fetch(f"{REC}/SHA256SUMS.q16k", work / "SHA256SUMS.q16k")
    out = _verify(dest, _read_sums(work / "SHA256SUMS.q16k"), 5)
    prefix.unlink(missing_ok=True)
    return out


In [ ]:
# Reference. What the archives actually bind is the DECODED reference: the bases of the FASTA, upper-cased,
# SHA-256 a6e4a745... (DATASET.md). That is header-independent, so two different FASTA renderings of
# T2T-CHM13v2.0 both pass it. We fetch the file DATASET.md pins (NCBI GCA_009914755.4) and report every sum.
import hashlib, os
from pathlib import Path
%cd /content
Path('cohort').mkdir(exist_ok=True)
REF_URL = ('https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/914/755/'
           'GCA_009914755.4_T2T-CHM13v2.0/GCA_009914755.4_T2T-CHM13v2.0_genomic.fna.gz')
PINNED_BYTES = 3156259565
PINNED_SHA256 = '51ab7d3c2a375ecff53ee2a1be03e58d57cc1601c948a860c03a78d724f26e4c'
PINNED_MD5 = 'cd1e52ce400c027ed0b7ab4b9d613f5a'          # the unpacked NCBI FASTA; md5 as printed in DATASET.md
DECODED_SHA256 = 'a6e4a74516252f28ac61291c9e11eba9e9cd53a576e014b3dfd51e4816333080'   # its bases, upper-cased

def _md5(path, buf=8 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _decoded_sha256(path):
    """SHA-256 of the bases only, upper-cased: what open_v1 checks against every archive header."""
    h = hashlib.sha256()
    drop = bytes.maketrans(b'acgtn', b'ACGTN')
    with open(path, 'rb') as f:
        for line in f:
            if line[:1] != b'>':
                h.update(line.translate(drop, b'\r\n'))
    return h.hexdigest()

ref = Path('cohort/reference.fa')
if not ref.exists():
    !curl -fsSL -o t2t.fna.gz "$REF_URL"
    !gunzip -f t2t.fna.gz && mv t2t.fna cohort/reference.fa
n, h256, hmd5 = ref.stat().st_size, _sha256(ref), _md5(ref)
print(f'reference file  {n} B')
print(f'  sha256 {h256}')
print(f'  md5    {hmd5}')
print(f'  pinned {PINNED_BYTES} B, sha256 {PINNED_SHA256}, md5 {PINNED_MD5}')
same = (n, h256, hmd5) == (PINNED_BYTES, PINNED_SHA256, PINNED_MD5)
print('  this file is', 'the pinned NCBI FASTA' if same else 'a DIFFERENT rendering - the bases decide')
dec = _decoded_sha256(ref)
print(f'decoded reference (bases, upper-cased)')
print(f'  sha256   {dec}')
print(f'  expected {DECODED_SHA256}')
if dec != DECODED_SHA256:
    raise SystemExit('this is not T2T-CHM13v2.0 as the archives were built against - they would refuse it')
print('reference OK: the decoded bases are what the archives bind')


In [ ]:
# Archives: fetched and verified; any empty file or wrong checksum stops here
paths = prepare_cohort(N_ASM, dest='/content/cohort', work='/content/dl')
print(f'{len(paths)} archives ready, names: {", ".join(p.name.replace(".q16k.rr3", "") for p in paths[:5])}'
      + (' ...' if len(paths) > 5 else ''))
!du -sh /content/cohort

In [ ]:
import json, time
import torch, panvram

t0 = time.perf_counter()
c = panvram.Cohort.open('/content/cohort', device='cuda', dataset='q16k')
torch.cuda.synchronize()
topen = time.perf_counter() - t0
print(f'OPEN\t{len(c.names)} assemblies\t{topen:.3f} s')
if len(c.names) != N_ASM:
    raise SystemExit(f'opened {len(c.names)} assemblies, expected {N_ASM}')
print('allocated', round(torch.cuda.memory_allocated() / 1e9, 3), 'GB; peak',
      round(torch.cuda.max_memory_allocated() / 1e9, 3), 'GB')
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

In [ ]:
out = {'tool': 'panvram-v1-q16k', 'device': 'cuda', 'gpu': torch.cuda.get_device_name(0),
       'assemblies': len(c.names), 'open_s': topen, 'batch': BATCH, 'runs': RUNS, 'rows': []}
for W in WS:
    g = torch.Generator(device='cuda').manual_seed(SEED)
    for _ in range(3):
        c.sample(BATCH, W, generator=g)
    torch.cuda.synchronize()
    ts = []
    for _ in range(RUNS):
        gg = torch.Generator(device='cuda').manual_seed(SEED)
        torch.cuda.synchronize()
        a = time.perf_counter()
        x = c.sample(BATCH, W, generator=gg)
        torch.cuda.synchronize()
        ts.append(time.perf_counter() - a)
        assert x.shape == (BATCH, W) and x.is_cuda
    ts.sort()
    med = ts[len(ts) // 2]
    print(f'A-GPU\tW {W}\tbatch {BATCH}\tmedian {med*1e3:.3f} ms\t{BATCH/med:.0f} windows/s'
          f'\t{BATCH*W/med/1e9:.3f} GB/s\tmin {min(ts)*1e3:.3f}\tmax {max(ts)*1e3:.3f}')
    out['rows'].append({'W': W, 'median_s': med, 'min_s': min(ts), 'max_s': max(ts),
                        'windows_per_s': BATCH / med, 'gb_per_s': BATCH * W / med / 1e9})
print(json.dumps(out, indent=1))

In [ ]:
# Correctness on the card: one batch of windows must equal the CPU decoder's answer for the same coordinates
W = WS[0]
g = torch.Generator(device='cuda').manual_seed(SEED)
x, coords = c.sample(256, W, generator=g, return_coords=True)
cc = panvram.Cohort.open('/content/cohort', device='cpu', dataset='q16k')
bad = 0
for i in range(coords.shape[0]):
    a, ci, s, _ = [int(v) for v in coords[i]]
    y = cc.fetch(cc.names[a], cc._core.contigs(a)[ci][0], s, W)
    if not torch.equal(x[i].cpu(), y):
        bad += 1
print(f'CHECK\tW {W}\twindows {coords.shape[0]}\tidentical {coords.shape[0]-bad}\tdiffer {bad}')
if bad:
    raise SystemExit('the GPU and CPU decoders disagree')